In [1]:
import sys
import os

# Add the parent directory (src) to the system path
# The '..' tells it to look one folder up from where the notebook is currently running
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [2]:
# Get the data
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [3]:
#Load the document and build a minisearch index for it
from ingest import  build_index
from evaluation_helper import get_documents


documents = get_documents(filer="llm-zoomcamp")
index = build_index(documents)

In [4]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [5]:
# Compute the relevance of the docuemnt
def compute_relevance_text(doc):
    doc_id = doc["document"]
    results = text_search(query=doc["question"])

    relevance = []

    for result in results:
        relevance.append(int(result["id"] == doc_id))

    return relevance

In [6]:
#check for 1
relevance = compute_relevance_text(ground_truth[0])
relevance

[0, 1, 0, 0, 0]

In [7]:
#Compute for all the records
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

#Change the relevance computation to use a generic search function

def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [8]:
# Compute the relevance for all the records using a generic search function
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [9]:
ground_truth_sample = ground_truth[:20]

relevance_total = compute_relevance_total(ground_truth_sample, text_search)
relevance_total

  0%|          | 0/20 [00:00<?, ?it/s]

[[0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0]]

In [10]:
#Compute the relevance for all the ground truth documents
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/515 [00:00<?, ?it/s]

### Search Metrics: Hit rate and Mean Reciprocal Rank(MRR)

In [11]:
example = [
    [1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
]

In [18]:
#Hit
def hit_rate(example):
    cnt = 0
    for line in example:
        if 1 in line:
            cnt += 1
    return cnt/len(example)

In [19]:
print(f"Hit rate: {hit_rate(example):.2%}")

Hit rate: 93.33%


In [20]:
#MRR
total_score = 0.0
for line in example:
    for rank in range(len(line)):
        if line[rank] == 1:
            total_score += 1/(rank + 1)
            break
        
mrr = total_score / len(example)
print(f"MRR: {mrr:.4f}")

MRR: 0.8222


In [21]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [22]:
mrr(example)
# 0.822

0.8222222222222222

In [23]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [24]:
evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/515 [00:00<?, ?it/s]

{'hit_rate': 0.8563106796116505, 'mrr': 0.7422330097087377}

# Search Parameter Tuning

In [26]:
#Trying different boosting techiqueq for questions
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [27]:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

  0%|          | 0/515 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.9087378640776699, 'mrr': 0.8111326860841422}


  0%|          | 0/515 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.9184466019417475, 'mrr': 0.8073786407766987}


  0%|          | 0/515 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.8563106796116505, 'mrr': 0.7422330097087377}


  0%|          | 0/515 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.829126213592233, 'mrr': 0.7139805825242718}


  0%|          | 0/515 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.8077669902912621, 'mrr': 0.6903883495145631}


In [28]:
# Tuning answers and sections to see if we can improve the results
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [29]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(
                f"Evaluating question_boost={question_boost},"
                f" answer_boost={answer_boost},"
                f" section_boost={section_boost}..."
            )
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/515 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/515 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/515 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/515 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/515 [00:00<?, ?it/s]

In [30]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
3,1.0,2.0,0.1,0.957282,0.871165
19,2.0,4.0,0.2,0.957282,0.871165
35,5.0,10.0,0.5,0.957282,0.871165
4,1.0,2.0,0.2,0.957282,0.870777
18,2.0,4.0,0.1,0.955340,0.870453
34,5.0,10.0,0.2,0.955340,0.869482
20,2.0,4.0,0.5,0.953398,0.868867
33,5.0,10.0,0.1,0.953398,0.868220
8,1.0,4.0,0.5,0.959223,0.863430
7,1.0,4.0,0.2,0.955340,0.861586


The best combination weights answer twice as heavily as question, with almost no weight on section. So the data says the opposite of where we started. The answer text matters more for retrieval than the question text. The intuition was wrong, and we'd never have known without measuring it. This is exactly why we evaluate instead of guess.

The first three rows have the same relative weights:

`question : answer : section = 1 : 2 : 0.1`

## Tuning Work flow

Search parameters can look arbitrary. This includes field boosts, number of results, filters, and other settings. Evaluation gives us a way to compare settings with evidence.

Grid search is fine when there are only a few settings. For a larger parameter space, use a smarter search strategy. You can sample random combinations, use Bayesian optimization, or keep a validation split so you don't overfit the evaluation set.

For text search on our dataset, grid search takes about one second per combination. That makes it practical to try many options. When each evaluation takes minutes instead of seconds, grid search becomes too expensive. In those cases, use Bayesian optimization with a library like hyperopt. It explores the parameter space more efficiently by focusing on combinations that are likely to improve the metric.